# Day 29 — Monochrome
### #30DayChartChallenge | April 2026

**Every dot is one of them.**  Nine of the world's most critically endangered land vertebrates, drawn at one dot per surviving individual.

**Tool:** `ggplot2` · single-panel dot grid in pure ink-on-paper monochrome
**Data:** IUCN Red List 2023/2024 + species-specific 2024 census reports (see `data/day_29/species.csv` for line-by-line provenance).
**Author:** Sharfudeen Yasar Arafath


In [ ]:
library(ggplot2)
library(showtext)
library(sysfonts)
library(scales)
library(ragg)

font_add_google("Outfit",          "outfit")
font_add_google("Roboto Condensed","roboto_condensed")
font_add_google("JetBrains Mono",  "jetbrains")
showtext_auto()
showtext_opts(dpi = 300)

# Dark theme — matches Days 1–27
bg_col   <- "#0a0e17"
text_col <- "#e0e6f0"
subtext  <- "#9eb3c8"
accent   <- "#ff6b6b"   # used only for the highlight stroke + callout
grid_col <- "#1a2030"
dot_col  <- "#e0e6f0"

species <- read.csv("../../data/day_29/species.csv", stringsAsFactors = FALSE)
species <- species[order(species$count), ]
species_levels <- species$species

ncols    <- 64
gap_rows <- 2.6
dot_size <- 1.65

species$rows_needed <- ceiling(species$count / ncols)
species$band_top <- cumsum(c(0, head(species$rows_needed + gap_rows, -1)))

dots <- do.call(rbind, lapply(seq_len(nrow(species)), function(i) {
  n <- species$count[i]
  row_in_band <- (seq_len(n) - 1) %/% ncols
  col_in_band <- (seq_len(n) - 1) %%  ncols
  data.frame(
    species   = species_levels[i],
    is_rhino  = species_levels[i] == "Northern white rhino",
    x = col_in_band,
    y = -(species$band_top[i] + row_in_band),
    stringsAsFactors = FALSE
  )
}))
dots$species <- factor(dots$species, levels = species_levels)

# Two-part labels: big bold count, then species name on the same baseline
label_df <- data.frame(
  species  = species_levels,
  count    = species$count,
  count_lbl = formatC(species$count, big.mark = ",", format = "d"),
  name_lbl  = paste0(" · ", species_levels),
  y = -species$band_top + 1.9
)

# Annotation pointing at the two Northern white rhino dots
rhino_y <- -species$band_top[species_levels == "Northern white rhino"]
callout <- data.frame(
  x = 6.5,
  y = rhino_y - 0.2,
  label = "Both surviving rhinos are female.\nFunctional extinction is now arithmetic."
)

x_min <- -0.5
x_max <- ncols - 0.5
y_min <- -(max(species$band_top + species$rows_needed)) - 0.6
y_max <- 3.4

p <- ggplot() +
  # all dots
  geom_point(data = subset(dots, !is_rhino),
             aes(x = x, y = y),
             colour = dot_col, fill = dot_col, shape = 21,
             size = dot_size, stroke = 0) +
  # highlighted rhino dots — same fill, but with accent ring so they read as "the survivors"
  geom_point(data = subset(dots, is_rhino),
             aes(x = x, y = y),
             colour = accent, fill = dot_col, shape = 21,
             size = dot_size + 1.4, stroke = 0.9) +
  # callout placed inline on the same row as the two rhino dots
  annotate("segment",
           x = 2.4, xend = 4.6,
           y = rhino_y, yend = rhino_y,
           colour = accent, linewidth = 0.35) +
  annotate("text",
           x = 5.0, y = rhino_y,
           label = "Both surviving rhinos are female  ·  functional extinction is now arithmetic",
           hjust = 0, vjust = 0.5,
           family = "roboto_condensed", fontface = "italic",
           size = 3.5, colour = accent) +
  # species labels: huge count + smaller name
  geom_text(data = label_df,
            aes(x = -0.5, y = y, label = count_lbl),
            family = "jetbrains", fontface = "bold",
            size = 6.0, colour = text_col, hjust = 0) +
  geom_text(data = label_df,
            aes(x = 4.2, y = y, label = name_lbl),
            family = "outfit",
            size = 3.6, colour = subtext, hjust = 0) +
  scale_x_continuous(limits = c(x_min, x_max),
                     expand = expansion(mult = c(0, 0))) +
  scale_y_continuous(limits = c(y_min, y_max),
                     expand = expansion(mult = c(0, 0))) +
  coord_cartesian(clip = "off") +
  labs(
    title = "Every dot is one of them.",
    subtitle = paste0(
      "Nine of the world's most critically endangered land vertebrates,\n",
      "drawn at one dot per surviving individual."),
    caption = paste(
      "Sources: IUCN Red List 2023/2024; Ol Pejeta Conservancy; CIRVA/Sea Shepherd acoustic survey; Kadoorie Farm & Bawangling NNR; IUCN AsRSG;",
      "Ujung Kulon NP; Land of the Leopard NP/WWF; WCS Cross River census; Greater Virunga + Bwindi joint count.   #30DayChartChallenge · Day 29 · Sharfudeen Yasar Arafath",
      sep = "\n"),
    x = NULL, y = NULL
  ) +
  theme_minimal(base_family = "outfit", base_size = 13) +
  theme(
    plot.background  = element_rect(fill = bg_col, colour = NA),
    panel.background = element_rect(fill = bg_col, colour = NA),
    panel.grid       = element_blank(),
    axis.text        = element_blank(),
    axis.ticks       = element_blank(),
    plot.title       = element_text(family = "outfit", face = "bold",
                                    size = 28, colour = text_col,
                                    margin = margin(b = 6)),
    plot.subtitle    = element_text(family = "roboto_condensed",
                                    size = 12, colour = subtext,
                                    lineheight = 1.3,
                                    margin = margin(b = 22)),
    plot.caption     = element_text(family = "jetbrains", size = 6.8,
                                    colour = subtext, hjust = 0,
                                    lineheight = 1.5,
                                    margin = margin(t = 20)),
    plot.caption.position = "plot",
    plot.title.position   = "plot",
    plot.margin = margin(28, 32, 22, 32)
  )

total_rows <- max(species$band_top + species$rows_needed)
fig_h <- max(8.5, 1.4 + total_rows * 0.17)

agg_png("../../chart/day_29_monochrome.png",
        width = 12, height = fig_h, units = "in", res = 300,
        background = bg_col)
print(p)
invisible(dev.off())
cat("Saved chart/day_29_monochrome.png  (", round(fig_h, 1), "in tall)\n")
